In [1]:
import numpy as np
import pandas as pd

import string, re, faiss

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed
from sentence_transformers import SentenceTransformer

from peft import LoraConfig, TaskType, get_peft_model
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset

#### The goal is to build a small question-answering system over an exoplanet dataset using either retrieval-augmented generation (RAG) or Low-Rank Adaptation (LoRA) fine-tuning.

#### Load the exoplanet dataset. The dataset includes information on all known exoplanets, including planet name, host star name, mass, orbital period, radius, temperature, discovery year et cetera.

In [2]:
df = pd.read_csv('data/exoplanets.csv', index_col=0)

In [3]:
df.head(3)

,pl_name,hostname,discoverymethod,disc_year,disc_facility,pl_orbper,pl_rade,pl_bmasse,pl_eqt,pl_orbsmax,pl_orbeccen,st_teff,st_rad,st_mass,sy_dist,sy_vmag,ra,dec,rowupdate
0,TOI-4495 b,TOI-4495,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),2.566990,2.483000,7.7,1735.0,0.03957,0.078,6210.00,1.309000,1.247,132.214,9.531,286.362495,37.025990,2026-02-12
1,TOI-4511 b,TOI-4511,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),10.980910,2.753049,NaN,NaN,NaN,NaN,5418.77,1.001450,NaN,121.832,10.537,49.305279,15.501727,2026-04-23
2,TOI-4513 b,TOI-4513,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),9.760423,2.345876,NaN,NaN,NaN,NaN,5623.93,0.796628,NaN,102.609,10.696,18.090518,13.755080,2026-04-23


#### Remove spaces from planet and host star names and cast discovery years to integers.

In [4]:
# replace spaces with "-" in planet and host star names
df['pl_name'] = df['pl_name'].apply(lambda x: x.replace(" ", "-"))
df['hostname'] = df['hostname'].apply(lambda x: x.replace(" ", "-"))
# convert discovery years to int
df['disc_year'] = df['disc_year'].astype('Int32')

#### Create two text representations for each row, a verbose description for semantic retrieval and a compact summary for lexical retrieval. The plan is to evaluate two retrieval strategies that select the relevant rows before generation: semantic retrieval (using cosine similarity over sentence embeddings) and lexical retrieval (using Jaccard similarity over sentences).

In [5]:
# create verbose description of each entry
df['description'] = df.apply(lambda x: f"The planet {x.pl_name} orbits around the star {x.hostname}. It was discovered in {x.disc_year}. It has a mass of {x.pl_bmasse} Mass Earth Units. It has an orbital period of {x.pl_orbper} days. It has a radius of {x.pl_rade} Radius Earth Units. It has a temperature of {x.pl_eqt} K.", axis=1)
df['description'] = df['description'].apply(lambda x: x.replace("nan", "unknown"))

In [6]:
# create compact summary of each entry
df['summary'] = df.apply(lambda x: f"name {x.pl_name} | host_name {x.hostname} | discovery_year {x.disc_year} | mass {x.pl_bmasse} | orbital_period {x.pl_orbper} | radius {x.pl_rade} | temperature {x.pl_eqt}", axis=1)
df['summary'] = df['summary'].apply(lambda x: x.replace("nan", "unknown"))

In [7]:
df.head(3)['description'].tolist()

['The planet TOI-4495-b orbits around the star TOI-4495. It was discovered in 2026. It has a mass of 7.7 Mass Earth Units. It has an orbital period of 2.56699 days. It has a radius of 2.483 Radius Earth Units. It has a temperature of 1735.0 K.',
 'The planet TOI-4511-b orbits around the star TOI-4511. It was discovered in 2026. It has a mass of unknown Mass Earth Units. It has an orbital period of 10.98090968596 days. It has a radius of 2.75304932 Radius Earth Units. It has a temperature of unknown K.',
 'The planet TOI-4513-b orbits around the star TOI-4513. It was discovered in 2026. It has a mass of unknown Mass Earth Units. It has an orbital period of 9.76042253684 days. It has a radius of 2.34587608 Radius Earth Units. It has a temperature of unknown K.']

In [8]:
df.head(3)['summary'].tolist()

['name TOI-4495-b | host_name TOI-4495 | discovery_year 2026 | mass 7.7 | orbital_period 2.56699 | radius 2.483 | temperature 1735.0',
 'name TOI-4511-b | host_name TOI-4511 | discovery_year 2026 | mass unknown | orbital_period 10.98090968596 | radius 2.75304932 | temperature unknown',
 'name TOI-4513-b | host_name TOI-4513 | discovery_year 2026 | mass unknown | orbital_period 9.76042253684 | radius 2.34587608 | temperature unknown']

#### RAG strategy 1 (FAISS): create embeddings for verbose descriptions and define function to retrieve, using FAISS, the 2 entries with verbose descriptions closest to the query by cosine similarity.

In [9]:
# create and index vector embeddings
st_model = SentenceTransformer('all-MiniLM-L6-v2')
corpus = df['description'].tolist()
embeddings = st_model.encode(corpus, convert_to_numpy=True, show_progress_bar=True)

faiss.normalize_L2(embeddings)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, 'exoplanet.index')
np.save('exoplanet.npy', embeddings)

# retrieve relevant information
def retrieve_faiss(query, k=2):
    embedding = st_model.encode([query], convert_to_numpy=True, show_progress_bar=False)
    faiss.normalize_L2(embedding)
    dst, idx = index.search(embedding, k)

    return df.iloc[idx[0]]['description'].tolist()

Batches:   0%|          | 0/197 [00:00<?, ?it/s]

In [10]:
retrieve_faiss("Any planet discovered in 2020?")

['The planet MOA-2020-BLG-135L-b orbits around the star MOA-2020-BLG-135L. It was discovered in 2022. It has a mass of 11.3 Mass Earth Units. It has an orbital period of unknown days. It has a radius of unknown Radius Earth Units. It has a temperature of unknown K.',
 'The planet AT2021ueyL-b orbits around the star AT2021ueyL. It was discovered in 2025. It has a mass of 425.89006538 Mass Earth Units. It has an orbital period of 4168.69 days. It has a radius of unknown Radius Earth Units. It has a temperature of unknown K.']

#### RAG strategy 2 (Jaccard): define function to retrieve the 3 entries with compact summaries closest to the query by Jaccard similarity.

In [11]:
def jaccard_similarity(wordlist1, wordlist2):
    intersection = len(list(set(wordlist1).intersection(set(wordlist2))))
    union = len(list(set(wordlist1).union(set(wordlist2))))
    return float(intersection) / float(union)

def jaccard_similarity_distance(a, b):
    # convert to lowercase and remove punctuation
    a = a.lower().translate(str.maketrans(dict.fromkeys(string.punctuation)))
    b = b.lower().translate(str.maketrans(dict.fromkeys(string.punctuation)))
    return jaccard_similarity(a.split(), b.split())

# retrieve relevant information
def retrieve_jaccard(query, k=3):
    dst = df['summary'].apply(lambda x: jaccard_similarity_distance(query, x))
    idx = np.argsort(dst)[-k:]

    return df['summary'][idx].to_list()

In [12]:
retrieve_jaccard("Any planet discovered in 2020?")

['name OGLE-2017-BLG-0406L-b | host_name OGLE-2017-BLG-0406L | discovery_year 2020 | mass 130.0 | orbital_period unknown | radius unknown | temperature unknown',
 'name KMT-2016-BLG-2364L-b | host_name KMT-2016-BLG-2364L | discovery_year 2020 | mass 1250.0 | orbital_period unknown | radius unknown | temperature unknown',
 'name KMT-2016-BLG-2397L-b | host_name KMT-2016-BLG-2397L | discovery_year 2020 | mass 836.0 | orbital_period unknown | radius unknown | temperature unknown']

#### Improved RAG strategy 2 (Filtered Jaccard): with queries involving numeric ranges (i.e., "mass between 0.9 and 1.1"), lexical similarity alone can break down. Use a lightweight parser for range filtering and apply retrieval within the filtered candidate set.

In [13]:
def parse_constraints(query):
    q = query.lower()

    # helper: extract "between a and b" or "from a to b"
    def between(field_keys):
        if not any(k in q for k in field_keys):
            return None
        m = re.search(r"(between|from)\s*([-\d.]+)\s*(and|to|-)\s*([-\d.]+)", q)
        if m:
            a, b = float(m.group(2)), float(m.group(4))
            return (min(a,b), max(a,b))
        return None

    filters = {}

    # discovery date range
    yr = between(["discovered", "discovery year", "discovery years", "year", "years"])
    if yr is not None:
        filters["disc_year"] = yr

    # mass range
    mr = between(["mass", "masses"])
    if mr is not None:
        filters["pl_bmasse"] = mr

    # radius range
    rr = between(["radius"])
    if rr is not None:
        filters["pl_rade"] = rr

    # orbital period range
    porbr = between(["orbital period", "period"])
    if porbr is not None:
        filters["pl_orbper"] = porbr

    # temperature range
    tr = between(["temperature"])
    if tr is not None:
        filters["pl_eqt"] = tr

    return filters

def apply_filters_to_df(filters, df):
    dff = df.copy()
    for col in filters.keys():
        lo, hi = filters[col]
        dff = dff[dff[col].between(lo, hi, inclusive="both")]
    return dff

def dff_from_query(query, df):
    filters = parse_constraints(query)
    if not filters:
        return df
    return apply_filters_to_df(filters, df)

# retrieve relevant information
def retrieve_jaccard_filtered(query, k=3):

    dff = dff_from_query(query, df)
    dff = dff.reset_index(drop=True)

    dst = dff['summary'].apply(lambda x: jaccard_similarity_distance(query, x))
    idx = np.argsort(dst)[-k:]

    return dff['summary'][idx].to_list()

#### Use the flan-t5-large LLM.

In [14]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

#### Create an example for one-shot prompting.

In [15]:
example_faiss = (f"Context:\n\n{df['description'].to_list()[1]}\n\n"
            f"{df['description'].to_list()[0]}\n\n"
            f"Question:\n\nWhat do you know about {df['pl_name'].to_list()[0]}?\n\n"
            f"Answer:\n\n{df['description'].to_list()[0]}\n\n"
           )

In [16]:
print(example_faiss)

Context:

The planet TOI-4511-b orbits around the star TOI-4511. It was discovered in 2026. It has a mass of unknown Mass Earth Units. It has an orbital period of 10.98090968596 days. It has a radius of 2.75304932 Radius Earth Units. It has a temperature of unknown K.

The planet TOI-4495-b orbits around the star TOI-4495. It was discovered in 2026. It has a mass of 7.7 Mass Earth Units. It has an orbital period of 2.56699 days. It has a radius of 2.483 Radius Earth Units. It has a temperature of 1735.0 K.

Question:

What do you know about TOI-4495-b?

Answer:

The planet TOI-4495-b orbits around the star TOI-4495. It was discovered in 2026. It has a mass of 7.7 Mass Earth Units. It has an orbital period of 2.56699 days. It has a radius of 2.483 Radius Earth Units. It has a temperature of 1735.0 K.




In [17]:
example_jaccard = (f"Context:\n\n{df['summary'].to_list()[2]}\n\n"
            f"{df['summary'].to_list()[1]}\n\n"
            f"{df['summary'].to_list()[0]}\n\n"
            f"Question:\n\nWhat do you know about {df['pl_name'].to_list()[0]}?\n\n"
            f"Answer:\n\n{df['description'].to_list()[0]}\n\n"
           )

In [18]:
print(example_jaccard)

Context:

name TOI-4513-b | host_name TOI-4513 | discovery_year 2026 | mass unknown | orbital_period 9.76042253684 | radius 2.34587608 | temperature unknown

name TOI-4511-b | host_name TOI-4511 | discovery_year 2026 | mass unknown | orbital_period 10.98090968596 | radius 2.75304932 | temperature unknown

name TOI-4495-b | host_name TOI-4495 | discovery_year 2026 | mass 7.7 | orbital_period 2.56699 | radius 2.483 | temperature 1735.0

Question:

What do you know about TOI-4495-b?

Answer:

The planet TOI-4495-b orbits around the star TOI-4495. It was discovered in 2026. It has a mass of 7.7 Mass Earth Units. It has an orbital period of 2.56699 days. It has a radius of 2.483 Radius Earth Units. It has a temperature of 1735.0 K.




#### Define a function that retrieves the context according to the chosen strategy, builds a prompt, feeds it to the LLM and returns the answer.

In [19]:
def rag_reply(question, rag_strategy='jaccard_filtered'):

    if rag_strategy == 'faiss':
        context  = "\n\n".join(retrieve_faiss(question))
        example = example_faiss
    elif rag_strategy == 'jaccard':
        context  = "\n\n".join(retrieve_jaccard(question))
        example = example_jaccard
    else:
        context  = "\n\n".join(retrieve_jaccard_filtered(question))
        example = example_jaccard

    prompt = f"Question:\n\n{question}\n\nContext:\n\n{context}\n\nAnswer:\n\n"

    complete_prompt = example + prompt

    #print(f"=== BEGIN PROMPT ===\n{complete_prompt}\n=== END PROMPT ===")

    inputs = tokenizer(complete_prompt, return_tensors="pt")

    ilen = inputs['input_ids'].shape[1]
    if ilen > 512:
        print(f"WARNING: input length is {ilen} > 512.")
    
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.20,
        top_p=0.90,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

def test_rag(q, rag_strategy='jaccard_filtered'):
    if rag_strategy == 'faiss':
        print(f"{rag_reply(q, rag_strategy='faiss')}\n\n{"\n\n".join(retrieve_faiss(q))}\n")
    elif rag_strategy == 'jaccard':
        print(f"{rag_reply(q, rag_strategy='jaccard')}\n\n{"\n\n".join(retrieve_jaccard(q))}\n")
    else:
        print(f"{rag_reply(q, rag_strategy='jaccard_filtered')}\n\n{"\n\n".join(retrieve_jaccard_filtered(q))}\n")

In [20]:
set_seed(42) # for reproducibility

#### Test #1

In [21]:
q = "What is the radius of Kepler-1604-b? When was it discovered?"

print("----- FAISS -----\n")
test_rag(q, rag_strategy='faiss')
print("----- JACCARD -----\n")
test_rag(q, rag_strategy='jaccard')
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- FAISS -----



1.41 Radius Earth Units. It was discovered in 2016.

The planet Kepler-1604-b orbits around the star Kepler-1604. It was discovered in 2016. It has a mass of unknown Mass Earth Units. It has an orbital period of 0.683684256 days. It has a radius of 1.41 Radius Earth Units. It has a temperature of unknown K.

The planet Kepler-1695-b orbits around the star Kepler-1695. It was discovered in 2020. It has a mass of unknown Mass Earth Units. It has an orbital period of 4.7329 days. It has a radius of 1.11948901 Radius Earth Units. It has a temperature of unknown K.

----- JACCARD -----



Kepler-1604-b was discovered in 2016 and has a mass of unknown. It has an orbital period of 0.683684256 and a radius of 1.41.

name LkCa-15-b | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name LkCa-15-c | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name Kepler-1604-b | host_name Kepler-1604 | discovery_year 2016 | mass unknown | orbital_period 0.683684256 | radius 1.41 | temperature unknown

----- FILTERED JACCARD -----



Kepler-1604-b was discovered in 2016 and has a mass of unknown. It has an orbital period of 0.683684256 and a radius of 1.41.

name LkCa-15-b | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name LkCa-15-c | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name Kepler-1604-b | host_name Kepler-1604 | discovery_year 2016 | mass unknown | orbital_period 0.683684256 | radius 1.41 | temperature unknown



#### Test #2

In [22]:
q = "Can you give me an example of a planet discovered in year 2010? What is its temperature?"

print("----- FAISS -----\n")
test_rag(q, rag_strategy='faiss')
print("----- JACCARD -----\n")
test_rag(q, rag_strategy='jaccard')
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- FAISS -----



WD-0806-661-b

The planet WD-0806-661-b orbits around the star WD-0806-661. It was discovered in 2011. It has a mass of 2542.627256 Mass Earth Units. It has an orbital period of unknown days. It has a radius of unknown Radius Earth Units. It has a temperature of 340.0 K.

The planet TYC-0434-04538-1-b orbits around the star TYC-0434-04538-1. It was discovered in 2021. It has a mass of 1938.763 Mass Earth Units. It has an orbital period of 193.2 days. It has a radius of unknown Radius Earth Units. It has a temperature of unknown K.

----- JACCARD -----



The planet SR-12-AB-c was discovered in 2010 and has a mass of 4131.62. It has an orbital period of unknown and a radius of unknown. Its temperature is unknown.

name MOA-2009-BLG-319L-b | host_name MOA-2009-BLG-319L | discovery_year 2010 | mass 67.3 | orbital_period unknown | radius unknown | temperature unknown

name 2MASS-J04414489+2301513-b | host_name 2MASS-J04414489+2301513 | discovery_year 2010 | mass 2383.6 | orbital_period unknown | radius unknown | temperature unknown

name SR-12-AB-c | host_name SR-12-AB | discovery_year 2010 | mass 4131.62 | orbital_period unknown | radius unknown | temperature unknown

----- FILTERED JACCARD -----



The planet SR-12-AB-c was discovered in 2010. It has a mass of 4131.62 and an orbital period of unknown. It has an unknown temperature.

name MOA-2009-BLG-319L-b | host_name MOA-2009-BLG-319L | discovery_year 2010 | mass 67.3 | orbital_period unknown | radius unknown | temperature unknown

name 2MASS-J04414489+2301513-b | host_name 2MASS-J04414489+2301513 | discovery_year 2010 | mass 2383.6 | orbital_period unknown | radius unknown | temperature unknown

name SR-12-AB-c | host_name SR-12-AB | discovery_year 2010 | mass 4131.62 | orbital_period unknown | radius unknown | temperature unknown



#### Test #3

In [23]:
q = "Can you give me an example of a planet with mass between 0.9 and 1.1?"

print("----- FAISS -----\n")
test_rag(q, rag_strategy='faiss')
print("----- JACCARD -----\n")
test_rag(q, rag_strategy='jaccard')
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- FAISS -----



Kepler-9-c

The planet 1RXS-J160929.1-210524-b orbits around the star 1RXS-J160929.1-210524. It was discovered in 2008. It has a mass of 3000.0 Mass Earth Units. It has an orbital period of unknown days. It has a radius of unknown Radius Earth Units. It has a temperature of 1700.0 K.

The planet Kepler-9-c orbits around the star Kepler-9. It was discovered in 2010. It has a mass of 29.9 Mass Earth Units. It has an orbital period of 38.9853 days. It has a radius of 8.08 Radius Earth Units. It has a temperature of unknown K.

----- JACCARD -----



Kepler-374-c has a mass of 1.1 and an orbital period of 3.282807 with a radius of 1.0.

name EPIC-228836835-b | host_name EPIC-228836835 | discovery_year 2021 | mass unknown | orbital_period 0.728113 | radius 1.1 | temperature unknown

name Kepler-392-c | host_name Kepler-392 | discovery_year 2014 | mass unknown | orbital_period 10.423118 | radius 1.1 | temperature unknown

name Kepler-374-c | host_name Kepler-374 | discovery_year 2014 | mass unknown | orbital_period 3.282807 | radius 1.1 | temperature unknown

----- FILTERED JACCARD -----



Kepler-102-b has a mass of 1.1 and an orbital period of 5.286965. It has s a temperature of 857 and a radius of 0.46.

name Proxima-Cen-b | host_name Proxima-Cen | discovery_year 2016 | mass 1.055 | orbital_period 11.18465 | radius unknown | temperature 218.0

name AU-Mic-d | host_name AU-Mic | discovery_year 2023 | mass 1.053 | orbital_period 12.73596 | radius unknown | temperature unknown

name Kepler-102-b | host_name Kepler-102 | discovery_year 2014 | mass 1.1 | orbital_period 5.286965 | radius 0.46 | temperature 857.0



#### Test #4

In [24]:
q = "Which planets were discovered in 2025?"

print("----- FAISS -----\n")
test_rag(q, rag_strategy='faiss')
print("----- JACCARD -----\n")
test_rag(q, rag_strategy='jaccard')
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- FAISS -----



Kepler-139-f

The planet Kepler-139-f orbits around the star Kepler-139. It was discovered in 2025. It has a mass of 36.0 Mass Earth Units. It has an orbital period of 355.0 days. It has a radius of unknown Radius Earth Units. It has a temperature of unknown K.

The planet KMT-2023-BLG-0119L-b orbits around the star KMT-2023-BLG-0119L. It was discovered in 2025. It has a mass of 83.7 Mass Earth Units. It has an orbital period of unknown days. It has a radius of unknown Radius Earth Units. It has a temperature of unknown K.

----- JACCARD -----



WISPIT-2-b, KMT-2021-BLG-0736L-b and KMT-2019-BL G-0578L-B were discovered in 2025.

name WISPIT-2-b | host_name WISPIT-2 | discovery_year 2025 | mass 1541.46777395 | orbital_period unknown | radius unknown | temperature unknown

name KMT-2021-BLG-0736L-b | host_name KMT-2021-BLG-0736L | discovery_year 2025 | mass 21.0 | orbital_period unknown | radius unknown | temperature unknown

name KMT-2019-BLG-0578L-b | host_name KMT-2019-BLG-0578L | discovery_year 2025 | mass 381.3940884 | orbital_period unknown | radius unknown | temperature unknown

----- FILTERED JACCARD -----



WISPIT-2-b, KMT-2021-BLG-0736L-b and KMT-2019-BLg-0578L-B were discovered in 2025.

name WISPIT-2-b | host_name WISPIT-2 | discovery_year 2025 | mass 1541.46777395 | orbital_period unknown | radius unknown | temperature unknown

name KMT-2021-BLG-0736L-b | host_name KMT-2021-BLG-0736L | discovery_year 2025 | mass 21.0 | orbital_period unknown | radius unknown | temperature unknown

name KMT-2019-BLG-0578L-b | host_name KMT-2019-BLG-0578L | discovery_year 2025 | mass 381.3940884 | orbital_period unknown | radius unknown | temperature unknown



#### From these tests, FAISS often retrieves incorrect entries, which leads the model to hallucinate. Jaccard performs better, but it still struggles with range-based questions; the filtered version mitigates this.

#### As an alternative approach, evaluate a LoRA fine-tuned model. Due to hardware constraints, switch to the flan-t5-base LLM and train on a small subset of the original dataset.

In [25]:
model_base = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tokenizer_base = AutoTokenizer.from_pretrained("google/flan-t5-base")

output_dir = "lora-flan-t5-base-exoplanet"
save_dir = "lora-flan-t5-base-exoplanet/final"

In [26]:
df.groupby('disc_year').count()

,pl_name,hostname,discoverymethod,disc_facility,pl_orbper,pl_rade,pl_bmasse,pl_eqt,pl_orbsmax,pl_orbeccen,st_teff,st_rad,st_mass,sy_dist,sy_vmag,ra,dec,rowupdate,description,summary
disc_year,,,,,,,,,,,,,,,,,,,,
1992,2,2,2,2,2,0,2,0,2,1,0,0,2,2,0,2,2,2,2,2
1994,1,1,1,1,1,0,1,0,1,1,0,0,1,1,0,1,1,1,1,1
1995,1,1,1,1,1,0,1,0,1,1,1,1,1,1,1,1,1,1,1,1
1996,6,6,6,6,6,0,6,0,6,6,5,6,6,6,6,6,6,6,6,6
1997,1,1,1,1,1,0,1,0,0,1,0,0,1,1,1,1,1,1,1,1
1998,6,6,6,6,6,0,6,0,6,6,3,5,6,6,6,6,6,6,6,6
1999,13,13,13,13,13,1,13,0,12,13,10,11,13,13,13,13,13,13,13,13
2000,16,16,16,16,16,0,16,1,15,15,10,7,16,16,16,16,16,16,16,16
2001,12,12,12,12,12,1,12,0,11,12,11,10,12,12,12,12,12,12,12,12


In [27]:
time_window = [1992, 1996]

mask = (df["disc_year"] >= time_window[0]) & (df["disc_year"] <= time_window[1])
df_subset = df.loc[mask].copy()

idx0 = 0
idx1 = df_subset.shape[0]

In [28]:
df.shape

(6273, 21)

In [29]:
df_subset.shape

(10, 21)

#### Build a training dataset for fine-tuning.

In [30]:
# host star
df_train = df_subset[idx0:idx1][['pl_name', 'hostname']].copy()
df_train['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"What is the host star of planet {x.pl_name}?", axis=1)
df_train['hostname'] = df_subset[idx0:idx1].apply(lambda x: f"The host star of {x.pl_name} is {x.hostname}.", axis=1)
df_train.columns = ['question', 'answer']

# host star
df_test = df_subset[idx0:idx1][['pl_name', 'hostname']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"Which star does {x.pl_name} orbit?", axis=1)
df_test['hostname'] = df_subset[idx0:idx1].apply(lambda x: f"The host star of {x.pl_name} is {x.hostname}.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# discovery date
df_test = df_subset[idx0:idx1][['pl_name', 'disc_year']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"When was {x.pl_name} discovered?", axis=1)
df_test['disc_year'] = df_subset[idx0:idx1].apply(lambda x: f"Planet {x.pl_name} was discovered in year {x.disc_year}.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# discovery date
df_test = df_subset[idx0:idx1][['pl_name', 'disc_year']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"In which year was planet {x.pl_name} discovered?", axis=1)
df_test['disc_year'] = df_subset[idx0:idx1].apply(lambda x: f"Planet {x.pl_name} was discovered in year {x.disc_year}.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# orbital period
df_test = df_subset[idx0:idx1][['pl_name', 'pl_orbper']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"What is the orbital period of {x.pl_name}?", axis=1)
df_test['pl_orbper'] = df_subset[idx0:idx1].apply(lambda x: f"The orbital period of {x.pl_name} is {x.pl_orbper} days.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# orbital period
df_test = df_subset[idx0:idx1][['pl_name', 'pl_orbper']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"Period of planet {x.pl_name} in days?", axis=1)
df_test['pl_orbper'] = df_subset[idx0:idx1].apply(lambda x: f"The orbital period of {x.pl_name} is {x.pl_orbper} days.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# radius
df_test = df_subset[idx0:idx1][['pl_name', 'pl_rade']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"What is the radius of {x.pl_name}?", axis=1)
df_test['pl_rade'] = df_subset[idx0:idx1].apply(lambda x: f"The radius of {x.pl_name} is {x.pl_rade} Earth Radius Units.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# radius
df_test = df_subset[idx0:idx1][['pl_name', 'pl_rade']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"Radius of planet {x.pl_name} in Earth Radius Units?", axis=1)
df_test['pl_rade'] = df_subset[idx0:idx1].apply(lambda x: f"The radius of {x.pl_name} is {x.pl_rade} Earth Radius Units.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# mass
df_test = df_subset[idx0:idx1][['pl_name', 'pl_bmasse']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"What is the mass of {x.pl_name}?", axis=1)
df_test['pl_bmasse'] = df_subset[idx0:idx1].apply(lambda x: f"The mass of {x.pl_name} is {x.pl_bmasse} Earth Mass Units.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# mass
df_test = df_subset[idx0:idx1][['pl_name', 'pl_bmasse']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"Mass of planet {x.pl_name} in Earth Mass Units?", axis=1)
df_test['pl_bmasse'] = df_subset[idx0:idx1].apply(lambda x: f"The mass of {x.pl_name} is {x.pl_bmasse} Earth Mass Units.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# temperature
df_test = df_subset[idx0:idx1][['pl_name', 'pl_eqt']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"What is the temperature of {x.pl_name}?", axis=1)
df_test['pl_eqt'] = df_subset[idx0:idx1].apply(lambda x: f"The temperature of {x.pl_name} is {x.pl_eqt} K.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

# temperature
df_test = df_subset[idx0:idx1][['pl_name', 'pl_eqt']].copy()
df_test['pl_name'] = df_subset[idx0:idx1].apply(lambda x: f"Temperature of planet {x.pl_name} in K?", axis=1)
df_test['pl_eqt'] = df_subset[idx0:idx1].apply(lambda x: f"The temperature of {x.pl_name} is {x.pl_eqt} K.", axis=1)
df_test.columns = ['question', 'answer']
df_train = pd.concat([df_train, df_test], ignore_index=True)

df_train['answer'] = df_train['answer'].astype(str)

df_train['question'] = df_train['question'].replace("nan", "unknown", regex=True)
df_train['answer'] = df_train['answer'].replace("nan", "unknown", regex=True)

In [31]:
df_train.shape

(120, 2)

In [32]:
df_train.head(3)

,question,answer
0,What is the host star of planet 47-UMa-b?,The host star of 47-UMa-b is 47-UMa.
1,What is the host star of planet 16-Cyg-B-b?,The host star of 16-Cyg-B-b is 16-Cyg-B.
2,What is the host star of planet 55-Cnc-b?,The host star of 55-Cnc-b is 55-Cnc.


In [33]:
df_train = df_train.sample(df_train.shape[0], random_state=42)

In [34]:
df_train.head(3)

,question,answer
44,What is the orbital period of ups-And-b?,The orbital period of ups-And-b is 4.617033 days.
47,What is the orbital period of PSR-B1257+12-b?,The orbital period of PSR-B1257+12-b is 25.262...
4,What is the host star of planet ups-And-b?,The host star of ups-And-b is ups-And.


#### Fine-tune the model.

In [35]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}")
    
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ["q", "v"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = TaskType.SEQ_2_SEQ_LM
)

lora_model = get_peft_model(model_base, lora_config)

print_trainable_parameters(lora_model)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    learning_rate=1.0e-3,
    per_device_train_batch_size=32,
    num_train_epochs=200,
    logging_strategy="epoch",
    save_strategy="epoch",
    push_to_hub=False
)

train_dataset = Dataset.from_pandas(df_train[["question", "answer"]])
train_dataset = train_dataset.rename_columns({"question": "input", "answer": "target"})

max_input_length = 64
max_target_length = 128

def preprocess_batch(batch):
    inputs = tokenizer_base(batch["input"], truncation=True, padding="longest", max_length=max_input_length)
    targets = tokenizer_base(batch["target"], truncation=True, padding="longest", max_length=max_target_length)
    return {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "labels": targets["input_ids"],
    }

tokenized_train_dataset = train_dataset.map(preprocess_batch, batched=True, remove_columns=train_dataset.column_names)
tokenized_train_dataset.set_format(type="torch")

data_collator = DataCollatorForSeq2Seq(tokenizer_base, model=lora_model, label_pad_token_id=-100)

def compute_exact_match(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer_base.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels == -100, tokenizer_base.pad_token_id, labels)
    decoded_labels = tokenizer_base.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    matches = [int(p == l) for p, l in zip(decoded_preds, decoded_labels)]

    return {"exact_match": float(np.mean(matches))}

trainer = Seq2SeqTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=data_collator,
    compute_metrics=compute_exact_match,
)


trainable params: 884736 || all params: 248462592 || trainable%: 0.3560841867092814


Map:   0%|          | 0/120 [00:00<?, ? examples/s]

In [36]:
# start training
trainer.train()

Step,Training Loss
4,13.731200
8,11.353000
12,9.202400
16,7.339300
20,5.325100
24,3.557800
28,2.889400
32,2.350200
36,1.976000
40,1.848500


TrainOutput(global_step=800, training_loss=0.5037641315534711, metrics={'train_runtime': 5877.9468, 'train_samples_per_second': 4.083, 'train_steps_per_second': 0.136, 'total_flos': 676733460480000.0, 'train_loss': 0.5037641315534711, 'epoch': 200.0})

In [37]:
# switch from training mode to inference mode
lora_model.eval();

In [38]:
print("model.training =", lora_model.training)

model.training = False


In [39]:
# saves adapter weights + config
lora_model.save_pretrained(save_dir)

# tokenizer should be saved too
tokenizer_base.save_pretrained(save_dir)

('lora-flan-t5-base-exoplanet/final/tokenizer_config.json',
 'lora-flan-t5-base-exoplanet/final/special_tokens_map.json',
 'lora-flan-t5-base-exoplanet/final/spiece.model',
 'lora-flan-t5-base-exoplanet/final/added_tokens.json',
 'lora-flan-t5-base-exoplanet/final/tokenizer.json')

#### Define a function that build a prompt, feeds it to the fine-tuned LLM and returns the answer.

In [40]:
def lora_reply(question):

    prompt = f"input:\n\n{question}\n\n\n\target:\n\n"

    #print(f"=== BEGIN PROMPT ===\n{complete_prompt}\n=== END PROMPT ===")

    inputs = tokenizer_base(prompt, return_tensors="pt")

    ilen = inputs['input_ids'].shape[1]
    if ilen > 512:
        print(f"WARNING: input length is {ilen} > 512.")
    
    output = lora_model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.20,
        top_p=0.95,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer_base.decode(output[0], skip_special_tokens=True)

def test_lora(q):
    print(f"{lora_reply(q)}\n")

In [41]:
set_seed(42) # for reproducibility

#### Test #1

In [42]:
sample_row = df_subset.iloc[idx0]
q = f"Hello! What is the orbital period of {sample_row['pl_name']}?"

print("----- QUESTION -----\n")
print(q+"\n")
print("----- LORA -----\n")
test_lora(q)
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- QUESTION -----

Hello! What is the orbital period of 47-UMa-b?

----- LORA -----



The orbital period of 47-UMa-b is 1078.0 days.

----- FILTERED JACCARD -----



The orbital period of 47-UMa-b is 1078.0.

name OGLE-2017-BLG-1049L-b | host_name OGLE-2017-BLG-1049L | discovery_year 2020 | mass 1760.0 | orbital_period unknown | radius unknown | temperature unknown

name HD-238090-b | host_name HD-238090 | discovery_year 2020 | mass 6.89 | orbital_period 13.671 | radius unknown | temperature 469.6

name 47-UMa-b | host_name 47-UMa | discovery_year 1996 | mass 804.08 | orbital_period 1078.0 | radius unknown | temperature unknown



#### Test #2

In [43]:
sample_row = df_subset.iloc[idx0+1]
q = f"And the mass of {sample_row['pl_name']}?"

print("----- QUESTION -----\n")
print(q+"\n")
print("----- LORA -----\n")
test_lora(q)
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- QUESTION -----

And the mass of 16-Cyg-B-b?

----- LORA -----



The mass of 16-Cyg-B-b is 565.7374 Earth Mass Units.

----- FILTERED JACCARD -----



The host name of 16-Cyg-B-b is 16-CYG-B. It was discovered in 1996. It has an orbital period of 798.5. Its mass is 565.7374.

name LkCa-15-b | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name LkCa-15-c | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name 16-Cyg-B-b | host_name 16-Cyg-B | discovery_year 1996 | mass 565.7374 | orbital_period 798.5 | radius unknown | temperature unknown



#### Test #3

In [44]:
sample_row = df_subset.iloc[idx0+2]
q = f"What is the temperature {sample_row['pl_name']}?"

print("----- QUESTION -----\n")
print(q+"\n")
print("----- LORA -----\n")
test_lora(q)
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- QUESTION -----

What is the temperature 55-Cnc-b?

----- LORA -----



The temperature of 55-Cnc-b is unknown K.

----- FILTERED JACCARD -----



The temperature of 55-Cnc-b is unknown.

name 55-Cnc-B-b | host_name 55-Cnc-B | discovery_year 2025 | mass 3.5 | orbital_period 6.7994 | radius unknown | temperature unknown

name 55-Cnc-B-c | host_name 55-Cnc-B | discovery_year 2025 | mass 5.3 | orbital_period 33.747 | radius unknown | temperature unknown

name 55-Cnc-b | host_name 55-Cnc | discovery_year 1996 | mass 276.0 | orbital_period 14.651552 | radius unknown | temperature unknown



#### Test #4

In [45]:
sample_row = df_subset.iloc[idx0+3]
q = f"Tell me the mass of planet {sample_row['pl_name']}"

print("----- QUESTION -----\n")
print(q+"\n")
print("----- LORA -----\n")
test_lora(q)
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- QUESTION -----

Tell me the mass of planet tau-Boo-b

----- LORA -----



The mass of tau-Boo-b is 1373.0256 Earth Mass Units.

----- FILTERED JACCARD -----



The mass of the planet tau-Boo-b is 1373.0256. It has an orbital period of 3.3124568 and a radius of unknown.

name LkCa-15-c | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name LkCa-15-b | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name tau-Boo-b | host_name tau-Boo | discovery_year 1996 | mass 1373.0256 | orbital_period 3.3124568 | radius unknown | temperature unknown



#### Test #5

In [46]:
sample_row0 = df_subset.iloc[idx0+3]
sample_row1 = df_subset.iloc[idx0+4]
q0 = f"Which is the planet with the largest mass, {sample_row0['pl_name']} or {sample_row1['pl_name']}?"
q1 = f"Which is the planet with the smallest mass, {sample_row0['pl_name']} or {sample_row1['pl_name']}?"

In [47]:
print("----- QUESTION -----\n")
print(q0+"\n")
print("----- LORA -----\n")
test_lora(q0)
print("----- FILTERED JACCARD -----\n")
test_rag(q0, rag_strategy='jaccard_filtered')

----- QUESTION -----

Which is the planet with the largest mass, tau-Boo-b or ups-And-b?

----- LORA -----



The host planet of tau-Boo-b is tau-Oo.

----- FILTERED JACCARD -----



tau-Boo-b

name LkCa-15-c | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name ups-And-b | host_name ups-And | discovery_year 1996 | mass 218.531 | orbital_period 4.617033 | radius unknown | temperature unknown

name tau-Boo-b | host_name tau-Boo | discovery_year 1996 | mass 1373.0256 | orbital_period 3.3124568 | radius unknown | temperature unknown



In [48]:
print("----- QUESTION -----\n")
print(q1+"\n")
print("----- LORA -----\n")
test_lora(q1)
print("----- FILTERED JACCARD -----\n")
test_rag(q1, rag_strategy='jaccard_filtered')

----- QUESTION -----

Which is the planet with the smallest mass, tau-Boo-b or ups-And-b?

----- LORA -----



The host planet of tau-Boo-b is tau-Oo.

----- FILTERED JACCARD -----



ups-And-b

name LkCa-15-c | host_name LkCa-15 | discovery_year 2015 | mass unknown | orbital_period unknown | radius unknown | temperature unknown

name ups-And-b | host_name ups-And | discovery_year 1996 | mass 218.531 | orbital_period 4.617033 | radius unknown | temperature unknown

name tau-Boo-b | host_name tau-Boo | discovery_year 1996 | mass 1373.0256 | orbital_period 3.3124568 | radius unknown | temperature unknown



#### Test #6

In [49]:
q = f"Can you tell me the name of a planet with a mass between 1.0 and 4.0?"

print("----- QUESTION -----\n")
print(q+"\n")
print("----- LORA -----\n")
test_lora(q)
print("----- FILTERED JACCARD -----\n")
test_rag(q, rag_strategy='jaccard_filtered')

----- QUESTION -----

Can you tell me the name of a planet with a mass between 1.0 and 4.0?

----- LORA -----



The host planet of 4.0 is 1.0 and 4.0.

----- FILTERED JACCARD -----



Ross-508-b

name HD-40307-b | host_name HD-40307 | discovery_year 2009 | mass 4.0 | orbital_period 4.3123 | radius unknown | temperature unknown

name HD-164922-d | host_name HD-164922 | discovery_year 2020 | mass 4.0 | orbital_period 12.458 | radius unknown | temperature unknown

name Ross-508-b | host_name Ross-508 | discovery_year 2022 | mass 4.0 | orbital_period 10.77 | radius unknown | temperature unknown



#### The LoRA fine-tuned model seems to underperform the filtered Jaccard RAG approach, likely because it uses a smaller model and was trained on a reduced dataset.